# Homework: Learn Image Features Without Labels
## A small SimCLR implementation with a CNN and CIFAR-10
**SEAS-8525 | Dr. Elbasheer**

**Question:** Does contrastive training teach our CNN useful features?

You will run a complete implementation, inspect its outputs, and write a final reflection.
The goal is understanding the training process, not reproducing the original paper's accuracy.

**The system:** two augmented views → shared CNN encoder **f** → embedding **h** → MLP head **g** → projection **z** → NT-Xent.

After training: keep **f**, remove **g**, and evaluate **h** with a small classifier.

**Submit:** this notebook with outputs saved, plus the reflection in the final Markdown cell.
Run the **homework** configuration for submission. The **debug** configuration only checks the pipeline.


## 0. Configure the environment
### Google Colab Pro
1. Upload this `.ipynb` using **File → Upload notebook** and save your copy in Drive.
2. Choose **Runtime → Change runtime type → Python 3 → GPU**. A T4 or better is suitable; availability varies.
3. Run the package cell below. Keep Colab's existing matched **torch/torchvision** installation.
4. Run once with `MODE = "debug"`. Then change it to `"homework"`, restart the session, and run all cells.
5. Enable `USE_DRIVE = True` before the homework run if you want checkpoints to survive a runtime reset.

### Your own computer
Use a fresh Python 3.11 or 3.12 environment. Install a matched PyTorch/torchvision pair using the
[official PyTorch installer](https://pytorch.org/get-started/locally/) for your OS and GPU.
Then install `numpy matplotlib tqdm pillow jupyterlab ipykernel` and launch `jupyter lab`.
Select that environment's kernel. CPU is supported for debugging; the full assignment expects an NVIDIA GPU.

The accompanying README includes commands. Do not mix unrelated torch and torchvision versions.
If torchvision fails to import, fix that installation and restart the kernel before continuing.


In [4]:
# Install only lightweight notebook dependencies. Do not replace Colab's CUDA-enabled PyTorch.
import sys, subprocess, importlib.util
requirements = {"numpy": "numpy>=1.26,<3", "matplotlib": "matplotlib>=3.8,<4",
                "tqdm": "tqdm>=4.66,<5", "PIL": "pillow>=10,<13"}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Lightweight dependencies ready. Python:", sys.version.split()[0])


Lightweight dependencies ready. Python: 3.13.15


In [5]:
import os, json, time, random, copy, platform, sys, importlib.util
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
try:
    import torchvision
    from torchvision import datasets, transforms, models
except Exception as exc:
    raise RuntimeError("torchvision could not load. Use a matched torch/torchvision installation and restart the runtime.") from exc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU detected. Use debug mode, or select a Colab GPU and restart.")


PyTorch: 2.11.0+cu128 | torchvision: 0.26.0+cu128
Device: cuda
GPU: Tesla T4


## 1. Choose the run and checkpoint location
- **Debug:** 1,000 training images and two epochs. It checks shapes, loss, training, and evaluation.
- **Homework:** 45,000 unlabeled pretraining images and 50 epochs. A separate 5,000-image validation set is held out.
- The classifier uses **10,000 labeled images from the pretraining split**. Those labels never enter the contrastive loss.
- GPU runtime depends on your device. The training cell reports an estimate after the first epoch.

**Resuming:** keep the same settings and run name. After reconnecting, rerun from the beginning.
The training cell automatically restores the last completed epoch, optimizer, scheduler, and random states.
An interrupted partial epoch is repeated. Change `RUN_NAME` for a new experiment.


In [6]:
MODE = "debug"                 # Change to "homework" after the debug run succeeds.
USE_DRIVE = False               # Set True on Colab to keep checkpoints across runtime resets.
RUN_NAME = "baseline_v1"        # Use a new name when changing the experiment.
SEED = 42
BATCH_SIZE = 256 if MODE == "homework" else 64  # Number of SOURCE images, not views.
EPOCHS = 50 if MODE == "homework" else 2
PROBE_EPOCHS = 30 if MODE == "homework" else 3
TEMPERATURE = 0.5
LR = 1e-3
NUM_WORKERS = 0  # Notebook-safe default; data loads in the main process. GPU computation still works.
assert MODE in {"debug", "homework"}
if MODE == "homework" and DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU for homework mode. CPU users should choose debug mode.")

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("USE_DRIVE is only for Colab. Use a local output directory instead.")
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/SEAS8525_SimCLR")
else:
    OUTPUT_ROOT = Path("./simclr_outputs")
DATA_ROOT = Path("/content/cifar10_data") if IN_COLAB else Path("./data")
RUN_DIR = OUTPUT_ROOT / f"{MODE}_{RUN_NAME}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)
CONFIG = dict(mode=MODE, seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS,
              temperature=TEMPERATURE, lr=LR, architecture="cifar_resnet18_mlp512_128_v1")
config_path = RUN_DIR / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError("Existing run has different settings. Choose a new RUN_NAME.")
config_path.write_text(json.dumps(CONFIG, indent=2))
environment = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,
                   numpy=np.__version__, device=str(DEVICE),
                   gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None)
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print("Run:", RUN_DIR, "| source batch:", BATCH_SIZE, "| views:", 2*BATCH_SIZE)


Run: simclr_outputs/debug_baseline_v1 | source batch: 64 | views: 128


## 2. Meet CIFAR-10 and separate the data
CIFAR-10 contains small color photographs in ten classes. We preserve its official test set.
We split the official training set into 45,000 pretraining images and 5,000 validation images.
The split is stratified for evaluation balance, but **class labels never determine positive pairs or enter pretraining**.

The test set is only used after choosing classifier checkpoints on validation data.
The images stay at **32 × 32**, not the original ImageNet SimCLR input size of 224 × 224.


In [3]:
raw_train = datasets.CIFAR10(DATA_ROOT, train=True, download=True)
raw_test = datasets.CIFAR10(DATA_ROOT, train=False, download=True)
classes = raw_train.classes
targets = np.array(raw_train.targets)
rng = np.random.default_rng(SEED)
train_ids, val_ids, probe_ids = [], [], []
for label in range(10):
    ids = rng.permutation(np.flatnonzero(targets == label))
    train_part, val_part = ids[500:], ids[:500]
    if MODE == "debug": train_part, val_part = train_part[:100], val_part[:50]
    train_ids.extend(train_part.tolist()); val_ids.extend(val_part.tolist())
    probe_ids.extend(train_part[:(1000 if MODE == "homework" else 50)].tolist())
train_ids, val_ids, probe_ids = map(np.array, (train_ids, val_ids, probe_ids))
test_ids = np.arange(len(raw_test))
if MODE == "debug":
    test_labels = np.array(raw_test.targets)
    test_ids = np.concatenate([np.flatnonzero(test_labels == c)[:100] for c in range(10)])
assert not set(train_ids) & set(val_ids)
assert set(probe_ids).issubset(set(train_ids))
np.savez(RUN_DIR / "splits.npz", train=train_ids, validation=val_ids, probe=probe_ids, test=test_ids)
print(f"Pretrain: {len(train_ids):,} | labeled probe: {len(probe_ids):,} | validation: {len(val_ids):,} | test: {len(test_ids):,}")
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for c, ax in enumerate(axes.flat):
    idx = train_ids[targets[train_ids] == c][0]
    ax.imshow(raw_train[idx][0]); ax.set_title(classes[c]); ax.axis("off")
plt.suptitle("Labels are shown for us, not supplied to contrastive training")
plt.tight_layout(); plt.show()


NameError: name 'DATA_ROOT' is not defined

## 3. Component 1: augmentations create the supervision
- Sample two transformations independently from one image.
- These two views are the **positive pair** because their source index is the same.
- Views from different source images are negatives, even when they show the same category.

**Predict before running:** will two calls on the same photograph produce identical views?
We use crop, flip, color jitter, and grayscale. We omit blur for this small-image teaching configuration.
Augmentations must preserve useful content; inspect the views before trusting the training recipe.


In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
augment = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
evaluate_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class PairDataset(Dataset):
    def __init__(self, raw, indices, transform):
        self.raw, self.indices, self.transform = raw, indices, transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        image, _unused_class_label = self.raw[int(self.indices[i])]
        return self.transform(image), self.transform(image)  # No class label returned.

class LabeledDataset(Dataset):
    def __init__(self, raw, indices): self.raw, self.indices = raw, indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        image, label = self.raw[int(self.indices[i])]
        return evaluate_transform(image), label

def show_tensor(x):
    return (x.cpu()*torch.tensor(STD)[:,None,None]+torch.tensor(MEAN)[:,None,None]).clamp(0,1).permute(1,2,0)

pairs = PairDataset(raw_train, train_ids, augment)
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for row in range(3):
    v1, v2 = pairs[row]
    axes[row,0].imshow(raw_train[int(train_ids[row])][0])
    axes[row,1].imshow(show_tensor(v1)); axes[row,2].imshow(show_tensor(v2))
    for ax in axes[row]: ax.axis("off")
for ax, title in zip(axes[0], ["Original", "View A", "View B"]): ax.set_title(title)
plt.tight_layout(); plt.show()

def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)
loader_generator = torch.Generator().manual_seed(SEED)
pair_loader = DataLoader(pairs, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type=="cuda",
                         worker_init_fn=seed_worker, generator=loader_generator)


## 4. Components 2 and 3: CNN encoder f and MLP head g
We initialize **ResNet-18 from scratch**, not from pretrained ImageNet weights.
For 32-pixel images we replace its large initial convolution with a 3 × 3, stride-1 convolution,
and remove its initial max-pool. Residual blocks extract features; global average pooling produces **h**.

| Stage | Output per image | Role |
|---|---|---|
| CNN encoder f | h: 512 numbers | Reusable image embedding |
| Dense → ReLU → Dense head g | z: 128 numbers | Contrastive comparison |

Both views use **the same model instance and weights**. The head is not a category classifier.
We also retain a copy of the initial encoder for a fair untrained baseline.


In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = models.resnet18(weights=None)
        self.encoder.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.encoder.maxpool = nn.Identity()
        self.encoder.fc = nn.Identity()
        self.projector = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 128))
    def forward(self, x):
        h = self.encoder(x)
        return h, self.projector(h)

seed_everything(SEED)
model = SimCLR()
initial_encoder_state = copy.deepcopy(model.encoder.state_dict())
model = model.to(DEVICE)
model.eval()
with torch.no_grad():
    h, z = model(torch.zeros(2, 3, 32, 32, device=DEVICE))
assert h.shape == (2,512) and z.shape == (2,128)
print("Input: [2, 3, 32, 32] → h:", tuple(h.shape), "→ z:", tuple(z.shape))
print(model.projector)


## 5. Component 4: NT-Xent, written explicitly
**Normalized Temperature-scaled Cross Entropy** answers: which other view is this anchor's positive?

1. Concatenate views in this order: `[A0, A1, ..., B0, B1, ...]`.
2. Normalize **z** by its length. Remember week 1: the dot product of unit vectors is cosine similarity.
3. Divide similarities by temperature **τ**. Lower τ emphasizes the closest competitors.
4. Mask the diagonal: an anchor cannot choose itself.
5. Identify its partner by **position**, not its class label, and apply cross entropy.

For N source images there are 2N views. Each anchor has one positive and **2N − 2 negatives**.
The positive is included in the denominator. We average over all 2N anchors.


In [ ]:
def nt_xent(z_a, z_b, temperature=0.5):
    if temperature <= 0: raise ValueError("Temperature must be positive")
    n = z_a.shape[0]
    if n < 2 or z_b.shape != z_a.shape: raise ValueError("Need equal batches with at least two sources")
    # Use float32 for this similarity computation even when the CNN uses mixed precision.
    z = F.normalize(torch.cat([z_a, z_b], dim=0).float(), dim=1)
    logits = (z @ z.T) / temperature
    diagonal = torch.eye(2*n, dtype=torch.bool, device=z.device)
    logits = logits.masked_fill(diagonal, float("-inf"))
    partners = (torch.arange(2*n, device=z.device) + n) % (2*n)
    return F.cross_entropy(logits, partners)

# Meaningful checks: matching, shuffled partners, collapse, and finite gradients.
toy = torch.eye(4)
good = nt_xent(toy, toy)
wrong = nt_xent(toy, toy.roll(1, dims=0))
collapsed = nt_xent(torch.ones(4,4), torch.ones(4,4))
assert good < wrong
assert torch.allclose(collapsed, torch.tensor(np.log(7), dtype=torch.float32), atol=1e-6)
za, zb = torch.randn(4,8,requires_grad=True), torch.randn(4,8,requires_grad=True)
nt_xent(za,zb).backward()
assert torch.isfinite(za.grad).all() and torch.isfinite(zb.grad).all()
print(f"Matched loss: {good:.3f} | wrong partners: {wrong:.3f} | collapsed: {collapsed:.3f}")
print("For four sources, collapse assigns probability 1/7 to the positive.")

with torch.no_grad():
    demo = F.normalize(torch.cat([toy,toy]),dim=1)
    similarity = demo @ demo.T
plt.figure(figsize=(6,5)); plt.imshow(similarity, vmin=0, vmax=1, cmap="Blues")
plt.xticks(range(8), [f"A{i}" for i in range(4)]+[f"B{i}" for i in range(4)])
plt.yticks(range(8), [f"A{i}" for i in range(4)]+[f"B{i}" for i in range(4)])
plt.title("Toy cosine similarities: diagonal excluded from loss")
plt.colorbar(); plt.tight_layout(); plt.show()


## 6. Train both networks together
The optimizer receives **all model parameters**, including f and g. Each batch creates its own negatives.
We use AdamW and a cosine learning-rate schedule as a teaching configuration, rather than reproducing the
original large-batch optimizer recipe. Mixed precision saves GPU memory; NT-Xent itself stays in float32.

The checkpoint is saved after each completed epoch. Keep the configuration unchanged when resuming.
Only load checkpoints you created with this notebook.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
checkpoint_path = RUN_DIR / "last_checkpoint.pt"
history, start_epoch = [], 0
if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert ckpt["config"] == CONFIG, "Checkpoint configuration mismatch"
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    history, start_epoch = ckpt["history"], ckpt["epoch"] + 1
    initial_encoder_state = ckpt["initial_encoder"]
    torch.set_rng_state(ckpt["torch_rng"])
    random.setstate(ckpt["python_rng"])
    loader_generator.set_state(ckpt["loader_rng"])
    if DEVICE.type == "cuda" and ckpt["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    print("Resuming at epoch", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_start, total_loss, seen = time.perf_counter(), 0.0, 0
    progress = tqdm(pair_loader, desc=f"Pretrain {epoch+1}/{EPOCHS}")
    for view_a, view_b in progress:
        view_a, view_b = view_a.to(DEVICE, non_blocking=True), view_b.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type=="cuda"):
            _, z = model(torch.cat([view_a, view_b], dim=0))
        z_a, z_b = z.chunk(2)
        loss = nt_xent(z_a,z_b,TEMPERATURE)
        if not torch.isfinite(loss): raise RuntimeError("Nonfinite loss. Stop and inspect the configuration.")
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss += loss.item()*len(view_a); seen += len(view_a)
        progress.set_postfix(loss=f"{total_loss/seen:.3f}")
    scheduler.step()
    seconds = time.perf_counter()-epoch_start
    history.append(dict(epoch=epoch+1, loss=total_loss/seen, seconds=seconds))
    state = dict(config=CONFIG, model=model.state_dict(), optimizer=optimizer.state_dict(),
                 scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,
                 history=history, initial_encoder=initial_encoder_state,
                 torch_rng=torch.get_rng_state(), python_rng=random.getstate(),
                 loader_rng=loader_generator.get_state(),
                 cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=="cuda" else None)
    temporary = RUN_DIR / "checkpoint.tmp"
    torch.save(state, temporary); os.replace(temporary, checkpoint_path)
    (RUN_DIR / "training_history.json").write_text(json.dumps(history, indent=2))
    print(f"Epoch {epoch+1}: {seconds/60:.1f} min; estimated remaining {(EPOCHS-epoch-1)*seconds/60:.1f} min")

plt.figure(figsize=(7,4))
plt.plot([r['epoch'] for r in history], [r['loss'] for r in history], marker='o')
plt.xlabel("Epoch"); plt.ylabel("Mean NT-Xent"); plt.title("Contrastive pretraining")
plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(RUN_DIR / "pretraining_loss.png"); plt.show()
print("Total recorded training minutes:", round(sum(r['seconds'] for r in history)/60, 1))


### Pause and interpret
- Did the loss decline? Did it fluctuate? Both the batches and augmentations change across steps.
- A lower matching loss is useful evidence, but **does not by itself prove useful category features**.
- Do not compare raw loss values across different batch sizes as if they measured the same task.

We will now evaluate h, not z, using labels that were absent from the pretraining objective.


## 7. Remove the head and extract frozen features
We compare two encoders with identical architecture and initial weights:
1. **Untrained:** the encoder as it was before contrastive training.
2. **Contrastive:** the encoder after the required epochs.

Neither encoder is updated during linear evaluation. `eval()` also freezes batch-normalization statistics.
We cache normalized h features on CPU, so training the small classifiers is inexpensive.
Normalization is applied equally to both models. Evaluation images use deterministic transforms.


In [ ]:
random_encoder = SimCLR().encoder
random_encoder.load_state_dict(initial_encoder_state)
random_encoder = random_encoder.to(DEVICE)
trained_encoder = model.encoder
for encoder in (random_encoder, trained_encoder):
    encoder.eval()
    for parameter in encoder.parameters(): parameter.requires_grad_(False)

@torch.no_grad()
def extract_features(encoder, raw, indices):
    loader = DataLoader(LabeledDataset(raw, indices), batch_size=512,
                        num_workers=0, pin_memory=DEVICE.type=="cuda")
    features, labels = [], []
    for x,y in tqdm(loader, desc="Extract h", leave=False):
        features.append(F.normalize(encoder(x.to(DEVICE)),dim=1).cpu())
        labels.append(y)
    return torch.cat(features), torch.cat(labels)

features = {}
for name, encoder in [("Untrained",random_encoder),("Contrastive",trained_encoder)]:
    features[name] = {"train":extract_features(encoder,raw_train,probe_ids),
                      "val":extract_features(encoder,raw_train,val_ids)}
print("Feature matrices:", {k:tuple(v['train'][0].shape) for k,v in features.items()})


## 8. Train the same linear classifier on each representation
Each classifier is a single linear layer mapping 512 features to ten class scores.
Use the same seed, labeled images, epochs, optimizer, and selection rule for both encoders.
Select each classifier's best epoch using **validation accuracy**, then evaluate it on the test set once.
This comparison measures the usefulness of the features under this particular evaluation protocol.


In [ ]:
@torch.no_grad()
def classifier_accuracy(classifier, x, y):
    classifier.eval(); correct = 0
    for start in range(0,len(y),1024):
        predictions = classifier(x[start:start+1024].to(DEVICE)).argmax(1).cpu()
        correct += (predictions == y[start:start+1024]).sum().item()
    return correct/len(y)

def fit_probe(train, val):
    seed_everything(SEED)
    classifier = nn.Linear(512,10).to(DEVICE)
    opt = torch.optim.AdamW(classifier.parameters(), lr=1e-2, weight_decay=1e-4)
    gen = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TensorDataset(*train),batch_size=256,shuffle=True,generator=gen)
    best_accuracy, best_state, best_epoch = -1, None, 0
    curve=[]
    for epoch in range(PROBE_EPOCHS):
        classifier.train()
        for x,y in loader:
            opt.zero_grad(set_to_none=True)
            loss=F.cross_entropy(classifier(x.to(DEVICE)), y.to(DEVICE))
            loss.backward(); opt.step()
        accuracy=classifier_accuracy(classifier,*val); curve.append(accuracy)
        if accuracy > best_accuracy:
            best_accuracy, best_epoch = accuracy, epoch+1
            best_state={k:v.detach().cpu().clone() for k,v in classifier.state_dict().items()}
    classifier.load_state_dict(best_state)
    return classifier,curve,best_epoch

probes, probe_curves, chosen_epochs = {}, {}, {}
for name in features:
    probes[name],probe_curves[name],chosen_epochs[name] = fit_probe(features[name]['train'], features[name]['val'])
    print(name, "best validation accuracy:", f"{max(probe_curves[name]):.2%}", "at epoch",chosen_epochs[name])
    torch.save(probes[name].state_dict(), RUN_DIR / f"{name.lower()}_probe.pt")
for name,curve in probe_curves.items(): plt.plot(range(1,len(curve)+1),curve,label=name)
plt.xlabel("Classifier epoch"); plt.ylabel("Validation accuracy"); plt.legend()
plt.title("Same classifier training, different frozen features"); plt.tight_layout(); plt.show()


## 9. Final test evaluation
We have finished selecting checkpoints. Now use the test set for the final comparison.
**Do not tune hyperparameters based on these test results.** A higher trained-encoder accuracy supports
the claim that pretraining improved the features for this task; it is not guaranteed on a short run.


In [ ]:
results=[]; test_predictions={}
for name,encoder in [("Untrained",random_encoder),("Contrastive",trained_encoder)]:
    x,y=extract_features(encoder,raw_test,test_ids)
    accuracy=classifier_accuracy(probes[name],x,y)
    with torch.no_grad():
        pred=torch.cat([probes[name](chunk.to(DEVICE)).argmax(1).cpu() for chunk in x.split(1024)])
    test_predictions[name]=(pred,y)
    results.append(dict(encoder=name,validation_accuracy=max(probe_curves[name]),
                        test_accuracy=accuracy,selected_probe_epoch=chosen_epochs[name]))
    print(f"{name:12s} | validation {max(probe_curves[name]):.2%} | test {accuracy:.2%}")
gain=results[1]['test_accuracy']-results[0]['test_accuracy']
print(f"Contrastive improvement: {gain*100:+.2f} percentage points")
(RUN_DIR / "results.json").write_text(json.dumps(dict(config=CONFIG, results=results, improvement_pp=gain*100),indent=2))
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar([r['encoder'] for r in results],[r['test_accuracy']*100 for r in results],color=['#8192a7','#16876b'])
axes[0].set_ylabel('Test accuracy (%)'); axes[0].set_ylim(0,100)
pred,y=test_predictions['Contrastive']; confusion=torch.zeros(10,10,dtype=torch.int64)
for actual,guess in zip(y,pred): confusion[actual,guess]+=1
axes[1].imshow(confusion.numpy(),cmap='Blues')
axes[1].set_xticks(range(10),classes,rotation=90); axes[1].set_yticks(range(10),classes)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual'); axes[1].set_title('Contrastive encoder + linear classifier')
plt.tight_layout(); plt.savefig(RUN_DIR / 'evaluation.png'); plt.show()


## 10. Inspect nearest neighbors
For the same held-out validation query, retrieve training images with the most similar **h** vectors.
The query is never in the gallery. Labels are displayed for interpretation but do not determine the ranking.
These are different source images, so retrieval tests something beyond matching augmented copies.

Inspect both rows. Does contrastive training improve the neighbors? Can you find a failure?
Color, background, and shape may influence similarity. Do not claim the model understands an object just from one gallery.


In [ ]:
QUERY_POSITION = 0  # Change to inspect another validation image.
assert 0 <= QUERY_POSITION < len(val_ids)
fig,axes=plt.subplots(2,6,figsize=(12,5))
for row,name in enumerate(['Untrained','Contrastive']):
    gallery, gallery_y=features[name]['train']
    queries,query_y=features[name]['val']
    similarities=gallery @ queries[QUERY_POSITION]
    neighbors=similarities.topk(5).indices.tolist()
    axes[row,0].imshow(raw_train[int(val_ids[QUERY_POSITION])][0])
    axes[row,0].set_title(f"{name}\nQuery: {classes[int(query_y[QUERY_POSITION])]}")
    for col,index in enumerate(neighbors,1):
        axes[row,col].imshow(raw_train[int(probe_ids[index])][0])
        axes[row,col].set_title(f"{classes[int(gallery_y[index])]}\ncos={similarities[index]:.2f}")
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.savefig(RUN_DIR / 'neighbors.png'); plt.show()


## 11. Save and submit
- Confirm the completed run says **homework**, not debug.
- Keep the augmentation gallery, loss curve, validation curves, test comparison, and neighbor gallery visible.
- Fill in the reflection below in your own words, citing your actual results.
- In Colab use **File → Download → Download .ipynb**. Ensure outputs are included when saving.
- Submit the executed notebook. Model checkpoints are for resuming and do not need to be submitted unless requested.

**If something goes wrong:**
- No CUDA: select a GPU runtime and rerun from the beginning.
- GPU out of memory: use a smaller source batch (128), a new run name, and report this change. Do not mix runs.
- DataLoader worker cleanup error: feature extraction uses `num_workers=0` to avoid worker subprocesses.
  In an older copy, change that argument inside `extract_features` to zero and rerun that cell.
  You do not need to repeat pretraining. Existing DataLoader objects are not changed by assigning a new global value.
- Colab reset: remount Drive and rerun with the same configuration to restore the last complete epoch.
- Low accuracy: verify full training completed and inspect augmentations. Report honestly; do not repeatedly tune on test data.

**Optional extension:** repeat pretraining without color jitter and grayscale using a new run name.
Keep the splits and evaluation protocol fixed. Discuss how the augmentation choice changes what features are learned.


In [ ]:
print("Submission mode:", MODE)
print("Completed epochs:", len(history), "/", EPOCHS)
print("Saved artifacts:", RUN_DIR)
print("Reflection still needs to be completed by the student.")
if MODE != "homework": print("DEBUG RUN ONLY: switch to homework mode before preparing the submission.")


## 12. Student reflection (300-500 words)
**Name:**  
**GPU and configuration:**  
**Completed pretraining epochs and elapsed training time:**  
**Untrained encoder test accuracy:**  
**Contrastive encoder test accuracy:**  

Replace this guidance with your own analysis:
1. Explain where the positive-pair targets came from and when class labels were first used to train a component.
2. Use your loss curve and encoder comparison to discuss whether useful features were learned. Distinguish matching loss from classification accuracy.
3. Explain the difference between h and z, why we trained the projection head, and why we removed it for evaluation.
4. Describe one retrieval or classification mistake you observed. Give a plausible explanation and acknowledge what the evidence cannot establish.



## Sources
- [Chen et al. (2020), SimCLR paper](https://proceedings.mlr.press/v119/chen20j.html)
- [Official SimCLR implementation](https://github.com/google-research/simclr)
- [CIFAR-10 dataset and attribution](https://www.cs.toronto.edu/~kriz/cifar.html)
- [PyTorch installation](https://pytorch.org/get-started/locally/)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

Since you are all embarking on your Praxis research phase soon, just note that this is an educational adaptation using a smaller encoder, small images, and a different optimizer.
It is not a reproduction of the original ImageNet benchmark.

Dr. Elbasheer
